<a href="https://colab.research.google.com/github/stella1298/AIFFEL_Quest_EPA/blob/master/NLP/NLP04/260908_Quest_Haena.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get install -qq -y fonts-nanum > /dev/null
!pip install -q sentencepiece   # Colab 에는 이미 있지만 확인용

In [2]:
import re, os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sentencepiece as spm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("SentencePiece:", spm.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.11.0+cu128
SentencePiece: 0.2.2
Device: cuda


In [6]:
!wget -q https://raw.githubusercontent.com/jungwon27/korean-parallel-corpus/master/korean-english-park.train.tar.gz
!tar -xzf korean-english-park.train.tar.gz
!find /content -name "korean-english-park.train.*"

tar (child): korean-english-park.train.tar.gz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now
/content/drive/MyDrive/Aiffel_EPA/s2s_translation/datasets/korean-english-park.train.ko
/content/drive/MyDrive/Aiffel_EPA/s2s_translation/datasets/korean-english-park.train.en


In [10]:
import glob

ko_files = glob.glob("/content/**/korean-english-park.train.ko", recursive=True)
en_files = glob.glob("/content/**/korean-english-park.train.en", recursive=True)

print("KO:", ko_files)
print("EN:", en_files)

ko_path = ko_files[0]
en_path = en_files[0]

KO: ['/content/drive/MyDrive/Aiffel_EPA/s2s_translation/datasets/korean-english-park.train.ko']
EN: ['/content/drive/MyDrive/Aiffel_EPA/s2s_translation/datasets/korean-english-park.train.en']


In [11]:
#ko_path = "korean-english-park.train.ko"
#en_path = "korean-english-park.train.en"

def preprocess_korean(sentence):
    sentence = sentence.strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r"[^가-힣a-zA-Z0-9?.!,-]+", " ", sentence)
    return re.sub(r"\s+", " ", sentence).strip()

def preprocess_english(sentence):
    sentence = sentence.strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r"[^a-zA-Z0-9?.!,-]+", " ", sentence)
    return re.sub(r"\s+", " ", sentence).strip()

In [12]:
cleaned_corpus = set()

with open(ko_path, encoding="utf-8") as f_ko, open(en_path, encoding="utf-8") as f_en:
    for ko, en in zip(f_ko, f_en):
        ko, en = ko.strip(), en.strip()
        if ko and en:
            cleaned_corpus.add((ko, en))

kor_corpus, eng_corpus = [], []

for ko, en in cleaned_corpus:
    ko = preprocess_korean(ko)
    en = preprocess_english(en)
    if len(ko.split()) <= 40 and len(en.split()) <= 40:
        kor_corpus.append(ko)
        eng_corpus.append(en)

assert len(kor_corpus) == len(eng_corpus)

print("문장 쌍:", len(kor_corpus))
print("한국어:", kor_corpus[:3])
print("영어:", eng_corpus[:3])

문장 쌍: 71911
한국어: ['미군은 성명에서 이라크 주둔 다국적군은 지난 2일 알-바드리의 은신처로 여겨지는 건물 4채에 폭격을 가했다 고 발표했다 .', '필리핀 지진-화산학 연구소의 벨라 투비아노사 박사는 불루산 화산에서 비교적 가까운 이로신 와 주반의 마을들을 잘 살피고 있다 며 화산 폭발이 일어날 가능성도 있다 고 설명했다 .', '올해 10월 예루살렘에서 열리는 평화를 위한 오케스트라 공연은 게르기예프의 생각이었다 .']
영어: ['The coalition forces called in close air support , killing al-Badri and the three others , the release said .', 'An ash explosion can occur there anytime , said Bella Tubianosa from the Philippine Institute of Volcanology and Seismology .', 'It was Gergiev s idea to take the World Orchestra for Peace to Jerusalem this October .']


In [13]:
with open("kor_corpus.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(kor_corpus))

with open("eng_corpus.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(eng_corpus))

In [14]:
VOCAB_SIZE = 10000
PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

spm.SentencePieceTrainer.train(
    input="kor_corpus.txt", model_prefix="ko_spm",
    vocab_size=VOCAB_SIZE, model_type="bpe",
    character_coverage=0.9995,
    pad_id=PAD_ID, bos_id=BOS_ID, eos_id=EOS_ID, unk_id=UNK_ID
)

spm.SentencePieceTrainer.train(
    input="eng_corpus.txt", model_prefix="en_spm",
    vocab_size=VOCAB_SIZE, model_type="bpe",
    character_coverage=1.0,
    pad_id=PAD_ID, bos_id=BOS_ID, eos_id=EOS_ID, unk_id=UNK_ID
)

True

In [15]:
encoder_tokenizer = spm.SentencePieceProcessor()
decoder_tokenizer = spm.SentencePieceProcessor()

encoder_tokenizer.load("ko_spm.model")
decoder_tokenizer.load("en_spm.model")

print("Encoder vocab:", encoder_tokenizer.get_piece_size())
print("Decoder vocab:", decoder_tokenizer.get_piece_size())

Encoder vocab: 10000
Decoder vocab: 10000


In [16]:
MAX_LEN = 50

class TranslationDataset(Dataset):
    def __init__(self, kor_corpus, eng_corpus):
        self.kor = kor_corpus
        self.eng = eng_corpus

    def __len__(self):
        return len(self.kor)

    def __getitem__(self, idx):
        src = encoder_tokenizer.encode(self.kor[idx])[:MAX_LEN]
        trg = decoder_tokenizer.encode(self.eng[idx])[:MAX_LEN-1]

        src = src + [PAD_ID] * (MAX_LEN - len(src))
        trg_input = [BOS_ID] + trg
        trg_label = trg + [EOS_ID]

        trg_input += [PAD_ID] * (MAX_LEN - len(trg_input))
        trg_label += [PAD_ID] * (MAX_LEN - len(trg_label))

        return torch.tensor(src), torch.tensor(trg_input), torch.tensor(trg_label)

In [17]:
BATCH_SIZE = 64

dataset = TranslationDataset(kor_corpus, eng_corpus)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

src, trg_input, trg_label = next(iter(train_loader))

print("src:", src.shape)
print("trg_input:", trg_input.shape)
print("trg_label:", trg_label.shape)

src: torch.Size([64, 50])
trg_input: torch.Size([64, 50])
trg_label: torch.Size([64, 50])


In [18]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embed_dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        return self.rnn(embedded)

In [19]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.shape[0]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        enc = encoder_outputs.permute(1, 0, 2)

        energy = torch.tanh(self.W1(enc) + self.W2(hidden))
        score = self.v(energy).squeeze(2)

        return F.softmax(score, dim=1)

In [20]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embed_dim, hidden_dim, attention, dropout=0.1):
        super().__init__()
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, embed_dim)
        self.rnn = nn.GRU(embed_dim + hidden_dim, hidden_dim)
        self.fc_out = nn.Linear(embed_dim + hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, trg_input, hidden, encoder_outputs):
        embedded = self.dropout(self.embedding(trg_input.unsqueeze(0)))

        attn = self.attention(hidden.squeeze(0), encoder_outputs)

        context = torch.bmm(
            attn.unsqueeze(1),
            encoder_outputs.permute(1, 0, 2)
        ).permute(1, 0, 2)

        rnn_input = torch.cat((embedded, context), dim=2)
        output, hidden = self.rnn(rnn_input, hidden)

        output = output.squeeze(0)
        embedded = embedded.squeeze(0)
        context = context.squeeze(0)

        prediction = self.fc_out(
            torch.cat((output, context, embedded), dim=1)
        )

        return prediction, hidden, attn

In [21]:
class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg=None, max_len=50):
        encoder_outputs, hidden = self.encoder(src)

        batch_size = src.shape[1]
        outputs, attentions = [], []

        if trg is not None:
            for t in range(trg.shape[0]):
                output, hidden, attn = self.decoder(
                    trg[t], hidden, encoder_outputs
                )
                outputs.append(output.unsqueeze(0))
                attentions.append(attn.unsqueeze(0))
        else:
            input = torch.full(
                (batch_size,), BOS_ID,
                dtype=torch.long,
                device=self.device
            )

            for _ in range(max_len):
                output, hidden, attn = self.decoder(
                    input, hidden, encoder_outputs
                )

                outputs.append(output.unsqueeze(0))
                attentions.append(attn.unsqueeze(0))

                input = output.argmax(1)

                if (input == EOS_ID).all():
                    break

        return torch.cat(outputs), torch.cat(attentions)

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EMBED_DIM = 256
HIDDEN_DIM = 512

encoder = Encoder(
    encoder_tokenizer.get_piece_size(),
    EMBED_DIM, HIDDEN_DIM
)

attention = BahdanauAttention(HIDDEN_DIM)

decoder = Decoder(
    decoder_tokenizer.get_piece_size(),
    EMBED_DIM, HIDDEN_DIM, attention
)

model = Seq2SeqAttention(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

print(model)

Seq2SeqAttention(
  (encoder): Encoder(
    (embedding): Embedding(10000, 256)
    (rnn): GRU(256, 512)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (decoder): Decoder(
    (attention): BahdanauAttention(
      (W1): Linear(in_features=512, out_features=512, bias=True)
      (W2): Linear(in_features=512, out_features=512, bias=True)
      (v): Linear(in_features=512, out_features=1, bias=False)
    )
    (embedding): Embedding(10000, 256)
    (rnn): GRU(768, 512)
    (fc_out): Linear(in_features=1280, out_features=10000, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)


In [23]:
def train_step(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for src, trg_input, trg_label in tqdm(loader, leave=False):
        src = src.permute(1, 0).to(device)
        trg_input = trg_input.permute(1, 0).to(device)
        trg_label = trg_label.permute(1, 0).to(device)

        optimizer.zero_grad()

        output, _ = model(src, trg_input)

        loss = criterion(
            output.reshape(-1, output.shape[-1]),
            trg_label.reshape(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
import time

EPOCHS = 10
loss_history = []
start_time = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()

    loss = train_step(model, train_loader, optimizer, criterion)
    loss_history.append(loss)

    epoch_time = time.time() - epoch_start
    total_time = time.time() - start_time

    print(f"Epoch {epoch+1:02d} | Loss: {loss:.4f} | "
          f"Epoch 시간: {epoch_time:.1f}초 | 누적: {total_time/60:.1f}분")

total_time = time.time() - start_time
print(f"\n총 학습 시간: {total_time/60:.2f}분 ({total_time:.1f}초)")

  0%|          | 0/1124 [00:00<?, ?it/s]

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.grid()
plt.show()

In [ ]:
def evaluate(sentence):
    model.eval()
    sentence = preprocess_korean(sentence)

    src_ids = encoder_tokenizer.encode(sentence)[:MAX_LEN]
    src_tokens = [encoder_tokenizer.id_to_piece(i) for i in src_ids]

    src_ids += [PAD_ID] * (MAX_LEN - len(src_ids))
    src = torch.tensor(src_ids).unsqueeze(1).to(device)

    with torch.no_grad():
        outputs, attentions = model(src, max_len=MAX_LEN)

    pred_ids = outputs.argmax(2).squeeze(1).tolist()

    if EOS_ID in pred_ids:
        pred_ids = pred_ids[:pred_ids.index(EOS_ID)]

    result = decoder_tokenizer.decode(pred_ids)
    trg_tokens = [decoder_tokenizer.id_to_piece(i) for i in pred_ids]

    attn = attentions.squeeze(1).cpu().numpy()
    attn = attn[:len(pred_ids), :len(src_tokens)]

    return result, src_tokens, trg_tokens, attn

In [ ]:
def translate(sentence):
    result, src_tokens, trg_tokens, attn = evaluate(sentence)

    print("입력 :", sentence)
    print("번역 :", result)

    if len(trg_tokens) == 0:
        return

    plt.figure(figsize=(10, 6))
    plt.imshow(attn, aspect="auto", cmap="viridis")
    plt.xticks(range(len(src_tokens)), src_tokens, rotation=90)
    plt.yticks(range(len(trg_tokens)), trg_tokens)
    plt.xlabel("Korean")
    plt.ylabel("English")
    plt.title("Bahdanau Attention")
    plt.tight_layout()
    plt.show()

In [ ]:
test_sentences = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다."
]

for sentence in test_sentences:
    translate(sentence)